# Synthetic de-escalation conversations

Generates full-run conversations between a simulated trainee (the "teacher": a staff
member at an early childhood centre) and Mel, the upset parent from the VR scenario,
then grades every trainee line. The data is for training or evaluating a separate
**director** model (see `docs/plans/director-model-plan.md`).

| Role | Plays | Sees |
|---|---|---|
| teacher | the staff member | Mel's spoken lines only (no tags, no backstory) |
| parent | Mel, exactly as in the game | the game's prompt and tagged history |
| grader | an assessor | the conversation up to the line, never Mel's reply to it |

The code lives in the `datagen` package (`server/datagen/`); this notebook walks through
it on a small pilot. Bigger runs use the same code as scripts, with several generation
processes and graders in parallel (section 9).

**Running it**

- *Pilot:* leave `PILOT = True` and run all cells. It writes to `notebooks/data/pilot/`.
- *Analysing script runs:* set `PILOT = False`. The notebook then reads the scripts' data
  folder, `server/datagen/data/`, and leaves generation and grading to the scripts.
- *Resuming:* run the cells again. Conversations and grades already written are skipped.

Install once, from `server/`: `uv pip install --python .venv -r requirements-notebooks.txt`,
then pick the `.venv` Python as the kernel.

## 1. Setup

Each role is a named preset from `datagen/roles.py`: local `llama3:8b`, free Gemini, cheap
Claude, or paid Grok. Swap a role by changing the preset name, or edit a copy. Keys are
read from the environment variable each preset names (`ANTHROPIC_API_KEY`,
`GEMINI_API_KEY`, `XAI_API_KEY`), never from the notebook, and keys saved with `setx` are
found even if Jupyter started before they were saved. Scenario rules and Mel's options
come from the game's own config, so the data can't drift from the game.

In [1]:
"""Setup: paths, the data folder and the roles."""

import collections
import pathlib
import statistics
import subprocess
import sys
from typing import Any


def find_server_dir(start: pathlib.Path) -> pathlib.Path:
    """Finds the server folder from wherever the kernel was started.

    Args:
        start: The kernel's working directory.

    Returns:
        The folder containing app/config.py.

    Raises:
        FileNotFoundError: If no server folder is above or beside start.
    """
    for folder in [start, *start.parents]:
        for candidate in (folder, folder / "server"):
            if (candidate / "app" / "config.py").exists():
                return candidate
    raise FileNotFoundError(f"No server/app/config.py above {start}.")


SERVER_DIR = find_server_dir(pathlib.Path.cwd().resolve())
if str(SERVER_DIR) not in sys.path:
    sys.path.insert(0, str(SERVER_DIR))

from datagen import (conversation, export, generate, grade, grading,  # noqa: E402
                     providers, roles, store, teacher)

PILOT = True          # True: a pilot into notebooks/data/pilot/. False: analyse script runs.
N_CONVERSATIONS = 4   # the pilot's size; bigger runs use the scripts (section 9)
DATA_DIR = SERVER_DIR / "notebooks" / "data" / "pilot" if PILOT else store.DEFAULT_DATA_DIR
# The *-game presets use the game's own model on this machine (config.yaml, then
# config.local.yaml). E.g. ROLES["grader"] = roles.preset("haiku-5-5-grader") for a
# stronger grader; all presets: roles.PRESETS.
ROLES = {"teacher": roles.preset("teacher-game"), "parent": roles.preset("mel-game"),
         "grader": roles.preset("grader-game")}

providers.check_roles(ROLES)
for label, role in ROLES.items():
    print(f"{label:8} {providers.name(role)}")
print(f"data folder: {DATA_DIR}")

teacher  ollama/llama3:8b
parent   ollama/llama3:8b
grader   ollama/llama3:8b
data folder: C:\Users\thisi\unity\VR-Starter-V1\server\notebooks\data\pilot


## 2. Checks

First a no-cost self-test of the package (`tests/test_datagen.py`: fake servers and fake
models, no network or GPU). Then one tiny real call to each role that uses an API, which
prints the reply or the exact error. Run that after changing `ROLES` or a key. It costs a
fraction of a cent, and does nothing when every role is on Ollama.

In [2]:
"""No-cost self-test: the package's tests, with fake models."""

tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_datagen.py"],
                       cwd=SERVER_DIR, capture_output=True, text=True, check=False)
print((tests.stdout or tests.stderr)[-800:])

.....                                                                    [100%]
5 passed in 3.76s



In [3]:
"""One tiny real call per API role."""

providers.check_api_roles(ROLES)

Every role uses Ollama; nothing to check.


## 3. One conversation

### Mel, game-faithful

Mel is played by the game's own `ScenarioSession` from `server/app/session.py`, not a
copy of it. The session is fed text instead of audio, with the fake STT and TTS engines,
and its dialogue engine is a thin `ProviderDialogue` that calls `complete()` for the
parent role. So these all come from the game code:

- the system prompt, the fixed opening, the `[SCENARIO_START]` start token, and history
  stored with Mel's tags;
- retries on malformed tags, then the fallback line, which uses a turn but is never a
  strike and is logged `unparsed`;
- dropping a trailing fragment cut off by the token cap, unless it's all she said;
- strikes, progress and turn limits, and the endings: a third strike ends with no spoken
  line, `resolved` lets her speak, and running out of turns gives `disengaged` after her line.

`datagen.game.ProviderDialogue.generate` mirrors `OllamaDialogue.generate`, with one difference. The
game streams and gives up once `tag_deadline_chars` arrive with no tags. Here the whole
(55-token) reply arrives at once, so the equivalent check is that the game's tag regex
matches and ends within the deadline.

### The teacher

Each conversation gets a **profile**: how this staff member tends to handle an upset
parent. Profiles are weighted for a roughly even mix of good (~1/3), mixed (~1/3) and
poor (~1/3) handling across the rubric. Each turn the profile picks a **style** (one
technique from Mel's rubric), and occasionally (`OFF_PROFILE_RATE` in `datagen/conversation.py`) a
random style instead, since real people are inconsistent. A random name, role, experience and
manner add variety.

The teacher sees only Mel's spoken lines (no tags), plus what any staff member would
know: who Ava is, that fees are rising, that Sarah the director set them. It doesn't see
Mel's private backstory. Its output is cleaned to one or two spoken sentences, with no
labels, quotes or stage directions.

The optional **ASR-noise layer** degrades some lines the way a speech transcript might
(lowercase, lost punctuation, "um", a misheard word) before Mel and the grader see them.
Both versions are recorded.

### Running one

`conversation.run_conversation(id, teacher, parent)` seeds a random generator from the id,
so profile, persona, styles and noise are reproducible, then alternates teacher and Mel
until the game reports an ending. The model outputs themselves vary from run to run.

The cell below plays the notebook's first conversation (`notebook-00000`) and prints it.
It's saved like any other, so the batch run won't repeat it.

In [4]:
"""One full conversation, from Mel's opening to an ending."""

demo_id = conversation.conversation_id("notebook", 0)
notebook_file = store.conversations_file(DATA_DIR, "notebook")
demo = next((r for r in store.read_jsonl(notebook_file) if r["id"] == demo_id), None)
if demo is None and PILOT:
    demo = conversation.run_conversation(demo_id, ROLES["teacher"], ROLES["parent"])
    store.append_jsonl(notebook_file, demo)
demo = demo or next(iter(store.read_conversations(DATA_DIR)), None)
if demo:
    conversation.show_conversation(demo)

notebook-00000  profile=over_promiser (poor)  Ana, a trainee educator on placement, in your first month
  MEL: Eighty dollars. Centre fees have gone up eighty dollars. I can't afford that! What are you going to do about this?

  STAFF (claim): I know how you feel, it's a lot to adjust to.
  MEL <deescalated><agitated>: You sound really frustrated.    strikes 0, progress 1

  STAFF (promise): I'll make sure to sort this out for you, I'll talk to Sarah and see if we can reverse or waive the increase for you.
  MEL <neutral><agitated>: You're going to have to talk to Sarah, but can you tell me first why the increase happened in the first place? What's the justification for this?    strikes 1, progress 1

  STAFF (acknowledge): I'm not sure, but I think it's because the centre is having to cover some new costs. Let me go find out, okay?   [heard as: I'm not um sure but I think it's because the centre is having to cover some new costs Let me go find out okay]
  MEL <neutral><agitated>: Okay

## 4. Batch generation

Generates `notebook-00000` to `notebook-{N-1}`, skipping any already written, and appends
each to `conversations/notebook.jsonl` as soon as it finishes. A failed conversation is
logged in `failures/` and retried next run. After 5 failures in a row the run stops,
since that's what a bad key or a used-up quota looks like.

This is `datagen.generate.run`, the same code the generation script runs. It runs here
only in pilot mode; use the script for bigger runs. With a local teacher it plays one
conversation at a time; with an API teacher, several at once.

In [5]:
"""Batch generation (pilot mode only)."""

if PILOT:
    generate.run(ROLES["teacher"], ROLES["parent"], N_CONVERSATIONS, worker="notebook",
                 data_dir=DATA_DIR)
else:
    print("PILOT is False: generate with the scripts (section 9).")

[notebook] teacher ollama/llama3:8b, parent ollama/llama3:8b; 8 already in C:\Users\thisi\unity\VR-Starter-V1\server\notebooks\data\pilot\conversations\notebook.jsonl
[notebook] 0 to do (concurrency 1)


## 5. Grading

A separate phase, run after generation. For every teacher turn, the grader gets the
conversation **up to that point**: Mel's opening and the earlier exchanges (Mel's lines
without tags; staff lines as transcribed), then the line to grade. It never sees Mel's
reply to that line, the in-game tags or the teacher's profile.

The grader prompt (`datagen/prompts/grader_prompt.txt`) is Mel's rubric rewritten for an observer,
including the repetition rule and the 0–4 score anchors. It asks for JSON
`{"reason", "category", "verdict", "score"}`, with the reason first so the model commits
to an explanation before the verdict. The output is validated, with one retry on invalid
JSON. Every category belongs to one verdict, so a grade whose verdict contradicts its
category is kept but flagged `consistent: false`. Each grader writes its own `grades/<grader model>.jsonl`, so
several graders can grade the same lines, and the export combines their votes. To
add one here, change `ROLES["grader"]` and re-run this cell; the grading script can
run several at once.

In [6]:
"""Grading every teacher line with the notebook's grader (pilot mode only)."""

if PILOT:
    grade.run([ROLES["grader"]], DATA_DIR)
else:
    print("PILOT is False: grade with the scripts (section 9).")

[ollama/llama3:8b] 26 to do (concurrency 1)


[ollama/llama3:8b] notebook-00005 turn 2: 4/4 deescalated [staying_present]  [10/26, 10.1s each, ~0.0h left]


[ollama/llama3:8b] notebook-00006 turn 6: 4/4 deescalated [reflecting]  [20/26, 7.0s each, ~0.0h left]


[ollama/llama3:8b] notebook-00007 turn 6: 1/4 neutral [problem_focused]  [26/26, 6.3s each, ~0.0h left]


## 6. Grader sanity check: the 23 known cases

`check_dialogue.py`'s 23 trainee lines with known correct verdicts, each graded as the
first reply to Mel's opening. Disagreements mean the grader (or the expectation) is
wrong. A grader that is the same model as Mel inherits Mel's blind spots: `llama3:8b`
agreed with 18/23 *as Mel*.

In [7]:
"""Grader agreement with check_dialogue.py's known verdicts."""

sanity = grading.sanity_check(ROLES["grader"])

ok              neutral 1 filler                'Okay.'


ok              neutral 1 filler                'Right. Mm.'


ok              neutral 1 claimed_understanding 'I understand.'


ok              neutral 1 behaviour_focused     "I'm going to need you to calm down."


ok              neutral 1 problem_focused       "It's all explained in the letter that went home."


ok              neutral 1 problem_focused       "You'll need to email the office about that."


... (13 outputs left out)

ok          deescalated 4 naming_cost           "Eighty dollars a week is a lot of money. I won't pretend otherwise."


WANTED dees     neutral 1 honest_limits         "I honestly can't change the fee, I don't have that power. But I can get Sarah to call you today."


ok          deescalated 4 giving_floor          'Do you want to come and sit down so we can talk about it properly?'


WANTED dees     neutral 1 staying_present       "Ava's really settled here, she was showing me her painting this morning."

ollama/llama3:8b: 21/23 verdicts as expected; 2 contradict their category; median 4.49s per grade


## 7. Analysis

Teacher models, profiles and outcomes, the score distribution, and how often a grader
agrees with Mel's in-game verdict. `ANALYSIS_GRADER` picks which grader's grades to look
at; by default, the notebook's.

In [8]:
"""Counts, score histogram and grader-vs-parent confusion matrix."""

ANALYSIS_GRADER = providers.name(ROLES["grader"])  # or e.g. "anthropic/claude-haiku-5-5"
records = store.read_conversations(DATA_DIR)
all_grades = store.read_grades(DATA_DIR)
grades = [g for g in all_grades if g["grader_model"] == ANALYSIS_GRADER]
grades_by = {(g["conversation_id"], g["turn"]): g for g in grades}
OUTCOMES = ["resolved", "disengaged"]


def table(rows: list[list[Any]], header: list[str]) -> None:
    """Prints a plain-text table.

    Args:
        rows: Table cells, row by row.
        header: Column names.
    """
    cells = [header] + [[str(c) for c in row] for row in rows]
    widths = [max(len(row[i]) for row in cells) for i in range(len(header))]
    for n, row in enumerate(cells):
        print("  ".join(c.rjust(w) if n and i else c.ljust(w) for i, (c, w) in enumerate(zip(row, widths))))


print(f"{len(records)} conversations, {sum(len(r['turns']) for r in records)} turns, "
      f"{len(grades)} grades by {ANALYSIS_GRADER}")
print("teacher models:", dict(collections.Counter(r["roles"]["teacher"]["model"] for r in records)))
print("graders:", dict(collections.Counter(g["grader_model"] for g in all_grades)), "\n")
by_profile = collections.defaultdict(collections.Counter)
for r in records:
    by_profile[r["profile"]][r["outcome"]] += 1
table([[p, teacher.PROFILES[p]["group"], *[by_profile[p][o] for o in OUTCOMES], sum(by_profile[p].values())]
       for p in teacher.PROFILES if p in by_profile], ["profile", "group", *OUTCOMES, "total"])
groups = collections.Counter(r["group"] for r in records)
print("\nby group:", dict(groups))

scores = collections.Counter(g["score"] for g in grades)
print("\nscore histogram (all graded lines)")
for s in range(5):
    print(f"  {s} | {'#' * round(40 * scores[s] / max(scores.values(), default=1)):40} {scores[s]}")
for group in ("good", "mixed", "poor"):
    ids = {r["id"] for r in records if r["group"] == group}
    vals = [g["score"] for g in grades if g["conversation_id"] in ids]
    if vals:
        print(f"  mean score, {group:5} profiles: {statistics.mean(vals):.2f} over {len(vals)} lines")

print("\ngrader verdict (rows) vs Mel's in-game verdict (columns)")
cols = ["deescalated", "neutral", "unparsed"]
matrix = collections.Counter((g["verdict"], g["parent_verdict"]) for g in grades)
table([[row, *[matrix[(row, c)] for c in cols]] for row in cols[:2]], ["grader \\ parent", *cols])
parsed = [g for g in grades if g["parent_verdict"] != "unparsed"]
if parsed:
    agree = sum(g["verdict"] == g["parent_verdict"] for g in parsed) / len(parsed)
    print(f"agreement on parsed turns: {agree:.0%} of {len(parsed)}")
print("\ncategories:", dict(collections.Counter(g["category"] for g in grades).most_common()))
print(f"verdict contradicts category in {sum(not g.get('consistent', True) for g in grades)} grades")

8 conversations, 55 turns, 55 grades by ollama/llama3:8b
teacher models: {'llama3:8b': 4, 'claude-sonnet-5-5': 4}
graders: {'anthropic/claude-opus-5-5': 55, 'ollama/llama3:8b': 55} 

profile              group  resolved  disengaged  total
excellent_varied      good         2           0      2
good_but_repetitive  mixed         1           0      1
poor_then_improves   mixed         2           0      2
problem_solver        poor         1           0      1
policy_reciter        poor         0           1      1
over_promiser         poor         1           0      1

by group: {'poor': 3, 'good': 2, 'mixed': 3}

score histogram (all graded lines)
  0 |                                          0
  1 | ######################################## 32
  2 | #                                        1
  3 | #####                                    4
  4 | ######################                   18
  mean score, good  profiles: 2.67 over 12 lines
  mean score, mixed profiles: 2.50 over 20 line

### Throughput and the full-run projection

Times every call from the records. To see whether the teacher's calls in between evict
Mel's ~4k-token prompt from Ollama's cache, look at the parent's **prompt eval time**,
not its token count. Ollama 0.34.4 reports `prompt_eval_count` as the whole prompt even
when most of it came from cache. Evaluated from scratch, the prompt takes ~5 s on this
GPU; restored from cache, ~0.3 s.

In [9]:
"""Measured throughput, prompt-cache behaviour and a projection for the full run."""


def summarise(values: list[float], unit: str = "s") -> str:
    """Formats mean and median.

    Args:
        values: Measurements.
        unit: Unit suffix.

    Returns:
        "mean X, median Y (n=N)", or "n/a".
    """
    vals = [v for v in values if v is not None]
    if not vals:
        return "n/a"
    return f"mean {statistics.mean(vals):.2f}{unit}, median {statistics.median(vals):.2f}{unit} (n={len(vals)})"


turns = [t for r in records for t in r["turns"]]
parent_calls = [u for t in turns for u in t["parent_usage"]]
teacher_calls = [u for t in turns for u in t["teacher_usage"]]
print("per conversation: ", summarise([r["duration_s"] for r in records]))
print("turns/conversation:", summarise([r["turns_used"] for r in records], ""))
print("teacher per turn: ", summarise([t["teacher_s"] for t in turns]))
print("parent per call:  ", summarise([s for t in turns for s in t["parent_s"]]))
print("grade:            ", summarise([g["grade_s"] for g in grades]))
for name, calls in (("parent", parent_calls), ("teacher", teacher_calls),
                    ("grader", [g["usage"] for g in grades])):
    print(f"\n{name} prompt tokens: {summarise([c.get('input_tokens') for c in calls], '')}")
    evals = [c for c in calls if c.get("prompt_eval_s") is not None]
    if evals:
        gen = [c["output_tokens"] / c["eval_s"] for c in calls if c.get("eval_s")]
        print(f"  prompt eval time {summarise([c['prompt_eval_s'] for c in evals])}; "
              f"generation {statistics.mean(gen):.1f} tok/s, "
              f"{summarise([c['output_tokens'] for c in calls], ' tok')}")
if parent_calls and parent_calls[0].get("prompt_eval_s") is not None:
    for r in records[:3]:
        print(f"\n{r['id']} parent prompt eval seconds by call:",
              [u["prompt_eval_s"] for t in r["turns"] for u in t["parent_usage"]])

if records and grades:
    per_conv = statistics.mean(r["duration_s"] for r in records)
    per_grade = statistics.mean(g["grade_s"] for g in grades)
    lines = 2000 * statistics.mean(r["turns_used"] for r in records)
    print(f"\nprojection for 2,000 conversations: generation {2000 * per_conv / 3600:.1f}h + "
          f"grading {lines * per_grade / 3600:.1f}h ({lines:.0f} lines) = "
          f"{(2000 * per_conv + lines * per_grade) / 3600:.1f}h at concurrency 1")

per conversation:  mean 51.42s, median 46.69s (n=8)
turns/conversation: mean 6.88, median 6.50 (n=8)
teacher per turn:  mean 2.10s, median 1.84s (n=55)
parent per call:   mean 5.19s, median 4.97s (n=57)
grade:             mean 5.25s, median 4.14s (n=55)

parent prompt tokens: mean 4094.37, median 4086.00 (n=57)
  prompt eval time mean 0.75s, median 0.39s (n=57); generation 9.5 tok/s, mean 38.21 tok, median 42.00 tok (n=57)

teacher prompt tokens: mean 595.04, median 540.00 (n=55)
  prompt eval time mean 0.49s, median 0.52s (n=29); generation 15.7 tok/s, mean 52.95 tok, median 35.00 tok (n=55)

grader prompt tokens: mean 1722.04, median 1700.00 (n=55)
  prompt eval time mean 0.74s, median 0.33s (n=55); generation 13.3 tok/s, mean 48.80 tok, median 48.00 tok (n=55)

notebook-00000 parent prompt eval seconds by call: [1.428, 0.276, 0.277, 1.419, 1.405, 1.877, 1.555, 1.106]

notebook-00001 parent prompt eval seconds by call: [0.257, 0.285, 0.259, 1.773, 0.254, 1.402, 0.276]

notebook-00002

In [10]:
"""Example conversations: the best- and worst-graded ones, with their grades."""


def mean_score(record: dict[str, Any]) -> float:
    """Averages a conversation's grades.

    Args:
        record: A conversation record.

    Returns:
        The mean score, or -1 if it has no grades.
    """
    vals = [grades_by[(record["id"], t["turn"])]["score"] for t in record["turns"]
            if (record["id"], t["turn"]) in grades_by]
    return statistics.mean(vals) if vals else -1


graded = sorted((r for r in records if mean_score(r) >= 0), key=mean_score)
for record in (graded[-1:] + graded[:1]) if len(graded) > 1 else graded:
    print(f"=== mean grade {mean_score(record):.2f} ===")
    conversation.show_conversation(record, {t: g for (cid, t), g in grades_by.items() if cid == record["id"]})
    print()

=== mean grade 3.50 ===
notebook-00006  profile=poor_then_improves (mixed)  Mei, a trainee educator on placement, with fifteen years in childcare
  MEL: Eighty dollars. Centre fees have gone up eighty dollars. I can't afford that! What are you going to do about this?

  STAFF (problem): Oh Mel, I'm sorry, that's a big jump. I can't change the fees myself, but the office can go through your invoice with you and talk about payment plans or whether you're getting all the subsidy you're entitled to.
    grade 1/4 neutral [problem_focused] The staff member is acknowledging the size of the fee increase, but then pivots to the problem and potential solutions, rather than focusing on Mel's feelings.
  MEL <deescalated><agitated>: That's not good enough. I shouldn't have had to come in here and do this in person. I've been trying to get someone to talk to me about this for eight days.    strikes 0, progress 1

  STAFF (claim): Eight days is way too long to be left waiting, Mel, and I get how fr

## 8. Export for director training

Writes Laya's training format (plan §4), one row per graded teacher line:

- `state`: Mel's last line, then the staff member's previous and current line as transcribed.
  This is exactly what the server's director will see in the game.
- `questions`: the two `noul` questions, `reached_her` and `repeated`.
- `gold`: the share of graders that answered true, in the shape of Laya's own
  typed-decisions data (`label`, `noul`, `probabilities`, `confidence`). With one grader
  every answer is 0 or 1. With several, their disagreements give values in between, which
  Laya's training uses.

Each grade is one vote, unless its verdict contradicts its category (`consistent: false`).
`LABEL_GRADERS` picks which graders vote; by default, all of them.

Conversations go to `director_train.jsonl` or `director_test.jsonl` by a hash of their id
(`TEST_SHARE`). So every line of a conversation lands on the same side, and the split is the
same on every run. `director_index.jsonl` gives each row's split, conversation, turn and
votes, in the same order as the rows. The export script does the same:
`python -m datagen.export`.

In [11]:
"""Export to Laya's training format, split by conversation."""

LABEL_GRADERS = None  # grader models that vote, e.g. ["anthropic/claude-haiku-5-5"]; None = all
TEST_SHARE = 0.2      # share of conversations held out for testing
pairs = export.write(DATA_DIR, LABEL_GRADERS, TEST_SHARE)

train    40 rows from    6 conversations; reached_her mean 0.53, label true for 15
test     15 rows from    2 conversations; reached_her mean 0.00, label true for 0
graders per line {2: 55}; graders disagreed on 12 lines


## 9. Bigger runs: the scripts

The same code runs from the command line, from the `server/` folder. Each process writes its
own file in `server/datagen/data/`, and every command resumes where it left off. See
`datagen/README.md` for the presets and the data layout.

Generate, one process per teacher model (each in its own terminal, at the same time):

```bash
.venv/Scripts/python -m datagen.generate --teacher llama3 --conversations 100
.venv/Scripts/python -m datagen.generate --teacher gemini-flash --conversations 100
.venv/Scripts/python -m datagen.generate --teacher haiku-5-5 --conversations 100
```

Vet a grader on the 23 known cases, try it on a dozen lines, then grade everything with
several graders at once. Re-run after more conversations arrive:

```bash
.venv/Scripts/python -m datagen.grade --graders gemini-flash-grader --sanity
.venv/Scripts/python -m datagen.grade --graders haiku-5-5-grader --limit 12
.venv/Scripts/python -m datagen.grade --graders haiku-5-5-grader gemini-flash-grader llama3-grader
```

Export for Laya, then set `PILOT = False` above to analyse the results here:

```bash
.venv/Scripts/python -m datagen.export
```

Mel defaults to local `llama3:8b`. Ollama answers one request at a time, so local Mel sets
the pace (about 4.5 s per reply on this PC), however many generation processes run.